In [ ]:
import os
import zipfile
import urllib.request
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство для обучения:', DEVICE)

In [ ]:
DATA_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l7/20writers.zip'
ARCHIVE_FILE = '20writers.zip'
DATA_DIR = 'writers'


def download_archive(url=DATA_URL, archive_name=ARCHIVE_FILE):
    """Скачивает архив с текстами, если его нет в рабочей папке."""
    if not os.path.exists(archive_name):
        urllib.request.urlretrieve(url, archive_name)


def extract_archive(archive_name=ARCHIVE_FILE, target_dir=DATA_DIR):
    """Распаковывает архив с корпусом писателей."""
    os.makedirs(target_dir, exist_ok=True)

    with zipfile.ZipFile(archive_name, 'r') as archive:
        archive.extractall(target_dir)


download_archive()
extract_archive()

print('Данные загружены и распакованы.')

In [ ]:
fragment_len = 1000    # длина текстового фрагмента
step = 100             # шаг сдвига окна


def read_text_file(path):
    """Читает текстовый файл. Если UTF-8 не подходит, пробует CP1251."""
    try:
        with open(path, encoding='utf-8') as file:
            return file.read()
    except UnicodeDecodeError:
        with open(path, encoding='cp1251') as file:
            return file.read()


def collect_text_fragments(folder, part_size=fragment_len, stride=step):
    """Собирает фрагменты текстов и числовые метки авторов."""
    fragments = []
    targets = []
    class_names = []

    for filename in sorted(os.listdir(folder)):
        full_path = os.path.join(folder, filename)

        if not os.path.isfile(full_path):
            continue

        author_name = os.path.splitext(filename)[0]
        class_names.append(author_name)
        author_id = len(class_names) - 1

        text = read_text_file(full_path)

        for start in range(0, len(text) - part_size, stride):
            finish = start + part_size
            fragments.append(text[start:finish])
            targets.append(author_id)

    return fragments, targets, class_names


texts, labels, author_names = collect_text_fragments(DATA_DIR)

for author in author_names:
    print(f'Автор: {author}')

print('Количество авторов:', len(author_names))
print('Количество фрагментов:', len(texts))

In [ ]:
max_words = 20000
max_len = 200

TOKEN_FILTERS = '!”#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r'


class SimpleTokenizer:
    """Простая замена Keras Tokenizer для текущей задачи."""

    def __init__(self, num_words=max_words, filters=TOKEN_FILTERS, lower=True):
        self.num_words = num_words
        self.filters = filters
        self.lower = lower
        self.word_index = {}

    def _split_text(self, text):
        if self.lower:
            text = text.lower()

        translation_table = str.maketrans({symbol: ' ' for symbol in self.filters})
        prepared_text = text.translate(translation_table)
        return prepared_text.split()

    def fit_on_texts(self, text_list):
        word_counter = Counter()
        first_seen = {}

        for text in text_list:
            for word in self._split_text(text):
                if word not in first_seen:
                    first_seen[word] = len(first_seen)
                word_counter[word] += 1

        ordered_words = sorted(
            word_counter.keys(),
            key=lambda word: (-word_counter[word], first_seen[word])
        )

        # Индекс 0 оставляем для заполнения пустых позиций
        self.word_index = {
            word: index + 1
            for index, word in enumerate(ordered_words[:self.num_words - 1])
        }

    def texts_to_sequences(self, text_list):
        sequences = []

        for text in text_list:
            sequence = []

            for word in self._split_text(text):
                word_id = self.word_index.get(word)

                if word_id is not None and word_id < self.num_words:
                    sequence.append(word_id)

            sequences.append(sequence)

        return sequences


def pad_sequences_left(sequences, max_length=max_len):
    """Повторяет поведение Keras pad_sequences с padding='pre', truncating='pre'."""
    result = np.zeros((len(sequences), max_length), dtype=np.int64)

    for row_index, sequence in enumerate(sequences):
        if len(sequence) == 0:
            continue

        trimmed = sequence[-max_length:]
        result[row_index, -len(trimmed):] = trimmed

    return result


def prepare_text_data(raw_texts, raw_labels):
    """Преобразует тексты в матрицу индексов, а метки — в массив классов."""
    tokenizer = SimpleTokenizer(num_words=max_words)
    tokenizer.fit_on_texts(raw_texts)

    sequences = tokenizer.texts_to_sequences(raw_texts)
    features = pad_sequences_left(sequences, max_length=max_len)
    targets = np.array(raw_labels, dtype=np.int64)

    return features, targets, tokenizer


X, y, tokenizer = prepare_text_data(texts, labels)

print('Форма X:', X.shape)
print('Форма y:', y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

In [ ]:
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 0.001


def make_loader(features, targets, batch_size=BATCH_SIZE, shuffle=False):
    """Создаёт загрузчик данных PyTorch."""
    x_tensor = torch.tensor(features, dtype=torch.long)
    y_tensor = torch.tensor(targets, dtype=torch.long)

    dataset = TensorDataset(x_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False
    )


train_loader = make_loader(X_train, y_train, shuffle=True)
test_loader = make_loader(X_test, y_test, shuffle=False)

print('Пакетов обучения:', len(train_loader))
print('Пакетов проверки:', len(test_loader))

In [ ]:
class WriterLSTM(nn.Module):

    def __init__(self, vocab_size, classes_count, embedding_dim=128, hidden_size=128):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.classifier = nn.Linear(hidden_size, classes_count)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden_state, _) = self.lstm(embedded)

        last_hidden = hidden_state[-1]
        logits = self.classifier(last_hidden)

        return logits


model = WriterLSTM(
    vocab_size=max_words,
    classes_count=len(author_names),
    embedding_dim=128,
    hidden_size=128
).to(DEVICE)

print(model)

In [ ]:
def run_epoch(model, data_loader, loss_function, optimizer=None):

    training_mode = optimizer is not None

    if training_mode:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in data_loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if training_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training_mode):
            logits = model(x_batch)
            loss = loss_function(logits, y_batch)

            if training_mode:
                loss.backward()
                optimizer.step()

        batch_size = y_batch.size(0)
        predictions = logits.argmax(dim=1)

        total_loss += loss.item() * batch_size
        correct += (predictions == y_batch).sum().item()
        total += batch_size

    average_loss = total_loss / total
    accuracy = correct / total

    return average_loss, accuracy


def train_model(model, train_loader, test_loader, epochs=EPOCHS, learning_rate=LEARNING_RATE):
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = {
        'loss': [],
        'accuracy': [],
        'val_loss': [],
        'val_accuracy': []
    }

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(
            model=model,
            data_loader=train_loader,
            loss_function=loss_function,
            optimizer=optimizer
        )

        val_loss, val_acc = run_epoch(
            model=model,
            data_loader=test_loader,
            loss_function=loss_function,
            optimizer=None
        )

        history['loss'].append(train_loss)
        history['accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        print(
            f'Эпоха {epoch:02d}/{epochs} | '
            f'loss: {train_loss:.4f} | accuracy: {train_acc:.4f} | '
            f'val_loss: {val_loss:.4f} | val_accuracy: {val_acc:.4f}'
        )

    return history

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE
)

In [ ]:
test_loss, test_acc = run_epoch(
    model=model,
    data_loader=test_loader,
    loss_function=nn.CrossEntropyLoss(),
    optimizer=None
)

print(f'Точность на тестовой выборке: {test_acc:.4f}')


my_text = """
Я шел по лесу и думал о смысле жизни. Ветер тихо шелестел листьями,
а солнце пробивалось сквозь кроны деревьев...
"""


def predict_writer(text, trained_model=model, fitted_tokenizer=tokenizer):
    """Определяет наиболее вероятного автора текста."""
    trained_model.eval()

    sequence = fitted_tokenizer.texts_to_sequences([text])
    features = pad_sequences_left(sequence, max_length=max_len)
    x_tensor = torch.tensor(features, dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits = trained_model(x_tensor)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]

    class_index = int(np.argmax(probabilities))
    return author_names[class_index], probabilities


predicted_author, prediction = predict_writer(my_text)

print('Нейросеть считает, что автор текста:', predicted_author)

In [ ]:
def show_training_plots(train_history):

    plt.figure()
    plt.plot(train_history['accuracy'])
    plt.plot(train_history['val_accuracy'])
    plt.title('Точность модели')
    plt.xlabel('Эпоха')
    plt.ylabel('Точность')
    plt.legend(['Обучающая выборка', 'Валидационная выборка'])
    plt.show()

    plt.figure()
    plt.plot(train_history['loss'])
    plt.plot(train_history['val_loss'])
    plt.title('Функция потерь')
    plt.xlabel('Эпоха')
    plt.ylabel('Потери')
    plt.legend(['Обучающая выборка', 'Валидационная выборка'])
    plt.show()


show_training_plots(history)